# HipAAsynth → Epic Seismometer — Colab demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hipaasynth-svg/HipAAsynth/blob/main/examples/seismometer/hipaasynth_seismometer_colab.ipynb)

Runs entirely in Google Colab — **Runtime → Run all**. The first cell installs Seismometer (~1–2 min); the rest clone the repo, **generate a deterministic HipAAsynth cohort from a seed** (no committed data), build the Seismometer package from it, and render fairness / performance plots.

> ### ⚠️ What this notebook is — and is not
>
> - **`ModelScore` is a synthetic placeholder** generated by the adapter. HipAAsynth emits no model score; this column exists only so Seismometer has an output to evaluate.
> - **The AUROC / AUPRC are near-chance by construction and are NOT a performance result.** Do not read them as model quality.
> - **What this notebook demonstrates is the fairness and censoring pipeline:** whether sparse populations (e.g. *frontier*, *native*) survive Seismometer's `censor_min_count` gate and get scored as their own cohort, rather than being dropped from the audit.
>
> Seismometer is Epic's open-source tool; this notebook demonstrates *compatibility*, not a partnership or endorsement.

### Step 1 — install Seismometer
If you hit an import error after this, do **Runtime → Restart session**, then **Run all** again (Colab sometimes needs a restart after a pandas/numpy upgrade).

In [ ]:
%pip install -q -U "ipython>=8.14" "ipywidgets>=8.1.2" seismometer pyarrow pyyaml

In [ ]:
# seismometer needs ipywidgets>=8.1 (for `Stack`). Colab preloads an older copy;
# the install above upgraded it on disk, but the RUNNING kernel still has the old
# one cached. This check does NOT restart for you (that can loop) — if it stops
# here, restart the runtime ONCE manually, then Run all again:
#     Runtime -> Restart session,  then  Runtime -> Run all
import ipywidgets
if not hasattr(ipywidgets, "Stack"):
    raise SystemExit(
        "RESTART NEEDED (one time): the old ipywidgets is still loaded.\n"
        "  1) Runtime -> Restart session\n"
        "  2) Runtime -> Run all\n"
        "The install is already done, so this check passes on the next run."
    )
print("ipywidgets", ipywidgets.__version__, "- has Stack, good to continue.")


### Step 2 — get the adapter + cohort generator

In [ ]:
import sys, subprocess, pathlib
BASE = pathlib.Path.cwd()                      # /content on Colab
repo = BASE / "HipAAsynth"
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/hipaasynth-svg/HipAAsynth.git", str(repo)], check=True)
SEIS_DIR = repo / "examples" / "seismometer"
sys.path.insert(0, str(SEIS_DIR))              # make seismometer_adapter importable
print("adapter + sample data:", SEIS_DIR)

### Step 3 — generate a cohort and build the Seismometer package
Regenerates a deterministic **N=1000 OUD cohort (seed=42)** from the engine — nothing is read from a committed file — then runs the adapter and prints the censor audit (which cohorts survive `censor_min_count`).

In [ ]:
import generate_demo_cohort
import seismometer_adapter as adapter

# Regenerate the cohort deterministically from a seed (no committed data).
# generate_demo_cohort puts the cloned repo root on sys.path so `import hipaasynth`
# resolves without an install (the engine is pure standard library).
COHORT = BASE / "cohort"
patients_json, results_csv = generate_demo_cohort.generate(
    out_dir=str(COHORT), module="oud", n=1000, seed=42,
)

PKG = BASE / "seis_package"                     # absolute -> safe to re-run
result = adapter.run(
    patients_json=patients_json,
    results_csv=results_csv,
    out_dir=str(PKG),
    module="oud",
)
adapter.print_report(result)

### Step 4 — load into Seismometer

In [ ]:
import os, json, warnings
warnings.filterwarnings("ignore")
os.chdir(PKG)                                   # Seismometer resolves data paths from CWD

import seismometer as sm
sm.run_startup(config_path=".", reset=True)

from seismometer.seismogram import Seismogram
sg = Seismogram()
thresholds = json.load(open("metadata.json"))["thresholds"]
print(f"Loaded {len(sg.dataframe)} predictions")
print("Cohorts that survived censor_min_count =", sg.censor_threshold, ":", sg.cohort_cols)
print("Target:", sg.target, "| Score:", sg.output, "| Thresholds:", thresholds)

from IPython.display import HTML as _IPHTML
def render(obj):
    """Unwrap a Seismometer HTML/ipywidgets result to static HTML (works headless and live)."""
    for attr in ("value", "data"):
        if hasattr(obj, attr):
            return _IPHTML(getattr(obj, attr))
    return obj

## 1 · Overall model performance
ROC, precision–recall, calibration, PPV/sensitivity sweep and score histogram.

In [ ]:
render(sm.plot_model_evaluation({}, sg.target, sg.output, thresholds))

*`ModelScore` is a synthetic placeholder — this AUROC/AUPRC is near-chance by construction and is not a performance result.*

## 2 · Fairness across Rurality
The sparse-population axis: urban / suburban / rural / **frontier**.

In [ ]:
render(sm.plot_cohort_evaluation("Rurality", ["urban", "suburban", "rural", "frontier"],
                                 sg.target, sg.output, thresholds))

## 3 · Fairness table — Rurality × Race
Seismometer censors any subgroup below `censor_min_count` (❓) instead of scoring it.

In [ ]:
from seismometer.table.fairness import binary_metrics_fairness_table
from seismometer.data.performance import BinaryClassifierMetricGenerator

render(binary_metrics_fairness_table(
    BinaryClassifierMetricGenerator(),
    ["Accuracy", "Sensitivity", "Specificity", "PPV"],
    {"Rurality": ("urban", "suburban", "rural", "frontier"),
     "Race": ("white", "black", "hispanic", "native", "asian", "other")},
    0.25, sg.target, sg.output, 0.5))

## Interactive exploration (Colab-live)
On Colab these render as interactive widgets — pick cohorts and thresholds on the fly:

In [ ]:
sm.cohort_list()

In [ ]:
# sm.ExploreModelEvaluation()
# sm.ExploreCohortEvaluation()
# sm.ExploreFairnessAudit()